In [26]:
import tensorflow as tf
import pandas as pd
import os

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
from scripts.resolve_split_paths import resolve_split

DATASET_ROOT = Path(os.environ["DATASET_ROOT"]).expanduser().resolve()
SPLIT_DIR = REPO_ROOT / "data"
PORTABLE_SPLIT_DIR = SPLIT_DIR / "portable"

for split_name in ("train_split.csv", "val_split.csv", "test_split.csv"):
    resolve_split(
        SPLIT_DIR / split_name,
        PORTABLE_SPLIT_DIR / split_name,
        DATASET_ROOT,
    )

In [ ]:
train_df = pd.read_csv(PORTABLE_SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(PORTABLE_SPLIT_DIR / "val_split.csv")
test_df = pd.read_csv(PORTABLE_SPLIT_DIR / "test_split.csv")

print("Loaded portable CSVs:")
print(train_df.head())

Loaded CSVs:
                                            filepath                    label  \
0  C:\Users\ASUS\Documents\8\comp258_f25\Final_Pr...           Pothole Issues   
1  C:\Users\ASUS\Documents\8\comp258_f25\Final_Pr...         Vandalism Issues   
2  C:\Users\ASUS\Documents\8\comp258_f25\Final_Pr...  Broken Road Sign Issues   
3  C:\Users\ASUS\Documents\8\comp258_f25\Final_Pr...   Illegal Parking Issues   
4  C:\Users\ASUS\Documents\8\comp258_f25\Final_Pr...         Vandalism Issues   

   label_idx  
0          5  
1          6  
2          0  
3          2  
4          6  


In [29]:
# ======================================================
# 2. Create the 'path' column (already exists, but refresh)
# ======================================================
train_df["path"] = train_df["filepath"]
val_df["path"]   = val_df["filepath"]
test_df["path"]  = test_df["filepath"]


In [30]:
# ======================================================
# 3. TensorFlow image parameters
# ======================================================
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [31]:
# ======================================================
# 4. Helper: Load & preprocess image
# ======================================================
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)   # NO img / 255.0img = img / 255.0
    return img, label

In [32]:
# ======================================================
# 5. Data Augmentation
# ======================================================
# data_augmentation = tf.keras.Sequential([
#     tf.keras.layers.RandomFlip("horizontal"),
#     tf.keras.layers.RandomRotation(0.10),
#     tf.keras.layers.RandomZoom(0.10),
# ])

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),         # flip left-right
    tf.keras.layers.RandomRotation(0.05),             # rotate ±5%
])
def augment(img, label):
    return data_augmentation(img), label

In [33]:
# ======================================================
# 6. Build TF datasets
# ======================================================
train_ds = tf.data.Dataset.from_tensor_slices((train_df["path"].values, train_df["label_idx"].values))
val_ds   = tf.data.Dataset.from_tensor_slices((val_df["path"].values,   val_df["label_idx"].values))
test_ds  = tf.data.Dataset.from_tensor_slices((test_df["path"].values,  test_df["label_idx"].values))


In [34]:
# ======================================================
# 7. Final pipelines
# ======================================================
train_ds = (
    train_ds
    .shuffle(3000)
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [35]:
val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


In [36]:
test_ds = (
    test_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [37]:
# ======================================================
# 8. Summary
# ======================================================
print("\nDatasets are ready!")
print("Train dataset:", train_ds)
print("Validation dataset:", val_ds)
print("Test dataset:", test_ds)


Datasets are ready!
Train dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
Validation dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
Test dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
